In [ ]:
'''The models included in this file are : Tesseract, EasyOCR, DocTR, PP-OCRv6, Florence-2, SuryaOCR, Nougat, Docling, PaddleOCR-VL-1.5, GOT-OCR2, DotsOCR, Qwen2.5-VL-3B,
DeepSeek-OCR, InternVL2-1B, MinerU2.5, HunyuanOCR'''


In [ ]:
#The dataset was downloaded as a csv file. To preserve the datatype of the image col (PIL image), the images are stored in a zip file.
#Loading the data
from zipfile import ZipFile
with ZipFile("/content/ClinOCR_images.zip") as z:
    z.extractall("/content/images")

from pathlib import Path
from PIL import Image
import pandas as pd

root = Path("/content")

df = pd.read_csv(root / "ClinOCR_Dataset.csv")
df["image"] = df["image"].apply(lambda p: Image.open(root /  p).convert("RGB"))

In [ ]:
#MODEL NAME : DotsOCR (Version : dots.mocr (released on 13 March 2026))
#PYTHON = "3.12"
'''torch==2.7.0
torchvision==0.22.0
torchaudio==2.7.0

transformers>=4.53
flash-attn==2.8.0.post2

datasets
pandas
numpy
Pillow
tqdm'''

In [ ]:
#Installation
!pip install -U pip
# PyTorch
!pip install \
torch==2.7.0 \
torchvision==0.22.0 \
torchaudio==2.7.0

# HuggingFace
!pip install \
transformers \
accelerate \
datasets \
huggingface_hub

# General
!pip install \
pandas \
numpy \
pillow \
tqdm

!pip install transformers==4.57.6


# Clone repository
!git clone https://github.com/rednote-hilab/dots.mocr.git

# Install package
%cd dots.mocr
!pip install -e .

In [ ]:
#Import block

import os
import gc
import json
import time
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import transformers
from dots_mocr.utils import dict_promptmode_to_prompt
from qwen_vl_utils import process_vision_info
from transformers import (AutoModelForCausalLM,AutoProcessor,)
warnings.filterwarnings("ignore")

In [ ]:
#Model Loading
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="dots-studio/dots.mocr",
    revision="e539fbb52280393adc081b289ec597430a0f9031"
)

In [ ]:
#Debugging and Patching
path = "/root/.cache/huggingface/hub/models--dots-studio--dots.mocr/snapshots/e539fbb52280393adc081b289ec597430a0f9031/modeling_dots_vision.py"

with open(path, "r") as f:
    text = f.read()

old = "from flash_attn import flash_attn_varlen_func"

new = """try:
    from flash_attn import flash_attn_varlen_func
except Exception:
    flash_attn_varlen_func = None"""

if old in text:
    text = text.replace(old, new)
    with open(path, "w") as f:
        f.write(text)
    print("Patched successfully!")
else:
    print("Import line not found.")


vision_file = "/root/.cache/huggingface/hub/models--dots-studio--dots.mocr/snapshots/e539fbb52280393adc081b289ec597430a0f9031/modeling_dots_vision.py"

with open(vision_file, "r") as f:
    text = f.read()

old = "hidden_states = hidden_states.bfloat16()"

new = "hidden_states = hidden_states.to(self.patch_embed.patchifier.proj.weight.dtype)"

if old not in text:
    raise ValueError("Couldn't find the bf16 line!")

text = text.replace(old, new)

with open(vision_file, "w") as f:
    f.write(text)

print("Patched successfully.")

In [ ]:
config.vision_config.attn_implementation = "sdpa"

model = AutoModelForCausalLM.from_pretrained(
    "dots-studio/dots.mocr",
    revision=revision,
    config=config,
    trust_remote_code=True,
    dtype=torch.float16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(
    "dots-studio/dots.mocr",
    revision=revision,
    trust_remote_code=True,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

print("Loaded!")

In [ ]:
#DotsOCR Function
def run_dotsocr(
    dataset,
    model,
    processor,
    output_dir="outputs/DotsOCR",
    save_images=True,
    overwrite=False,
    verbose=True,
):
    gc.collect()
    torch.cuda.empty_cache()

    raw_dir = os.path.join(output_dir, "raw")
    image_dir = os.path.join(output_dir, "images")

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(raw_dir, exist_ok=True)

    if save_images:
        os.makedirs(image_dir, exist_ok=True)

    results = []

    for idx, (_, sample) in enumerate(dataset.iterrows()):

        image_name = f"doc_{idx:05d}.png"
        raw_filename = f"doc_{idx:05d}.json"

        image_path = os.path.join(image_dir, image_name)
        raw_path = os.path.join(raw_dir, raw_filename)

        if (not overwrite) and os.path.exists(raw_path):
            if verbose:
                print(f"Skipping {image_name}")
            continue

        image = sample["image"]

        MAX_SIDE = 1024

        w, h = image.size
        scale = min(MAX_SIDE / w, MAX_SIDE / h, 1.0)

        if scale < 1:
            image = image.resize(
                (int(w * scale), int(h * scale)),
                Image.LANCZOS,
            )

        if save_images:
            image.save(image_path)

        success = True
        error = ""
        prediction = ""
        runtime = np.nan
        raw_output = {}

        try:

            gc.collect()
            torch.cuda.empty_cache()

            start = time.perf_counter()

            prompt = dict_promptmode_to_prompt["prompt_ocr"]

            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "image": image_path,
                        },
                        {
                            "type": "text",
                            "text": prompt,
                        },
                    ],
                }
            ]

            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )

            print("Pixel tensor shape:", inputs["pixel_values"].shape)

            inputs = inputs.to(model.device)

            if "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)

            if "pixel_values_videos" in inputs:
                inputs["pixel_values_videos"] = inputs["pixel_values_videos"].to(model.dtype)

            torch.cuda.empty_cache()

            generated_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
            )

            generated_ids_trimmed = [
                out_ids[len(in_ids):]
                for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            output_text = processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

            runtime = time.perf_counter() - start

            prediction = output_text[0]

            raw_output = {
                "prediction": prediction,
            }

        except Exception as e:
            print(f"\nError on sample {idx}:")
            raise

        try:
            with open(raw_path, "w", encoding="utf-8") as f:
                json.dump(
                    raw_output,
                    f,
                    indent=4,
                    ensure_ascii=False,
                    default=str,
                )
        except Exception:
            with open(
                raw_path.replace(".json", ".txt"),
                "w",
                encoding="utf-8",
            ) as f:
                f.write(str(raw_output))

        results.append(
            {
                "doc_id": sample["doc_id"],
                "image_name": image_name,
                "subset": sample["subset"],
                "template": sample["template"],
                "sample": sample["sample"],
                "ground_truth": sample["ground_truth"],
                "model": "DotsOCR",
                "prediction": prediction,
                "runtime_sec": runtime,
                "raw_output_file": raw_filename,
                "success": success,
                "error": error,
            }
        )

    results_df = pd.DataFrame(results)

    results_df.to_csv(
        os.path.join(output_dir, "results.csv"),
        index=False,
    )

    if verbose:
        print("\nFinished.")
        print(f"Processed : {len(results_df)} documents")
        print(f"Saved to : {output_dir}")

    return results_df

In [ ]:
#MODEL : HunyuanOCR
""""torch>=2.1.0, transformers>=4.57.0
accelerate>=0.34.0 sentencepiece>=0.1.99
tokenizers>=0.20.0, python == 3.12,
tensorboard>=2.15.0,tqdm>=4.66.0"""

In [ ]:
#Installation
pip install -U transformers accelerate sentencepiece safetensors

In [ ]:
import torch
import os
import time
from transformers import AutoProcessor, HunYuanVLForConditionalGeneration
from PIL import Image

In [ ]:
#Model loading
MODEL_ID = "tencent/HunyuanOCR"
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False
)

model = HunYuanVLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
).eval()

print("Model loaded!")

In [ ]:
#Hunyuan OCR function
PROMPT = (
    "提取文档图片中正文的所有信息用markdown格式表示，其中页眉、页脚部分忽略，"
    "表格用html格式表达，文档中公式用latex格式表示，按照阅读顺序组织进行解析。"
)

def run_hunyuanocr(df, processor, model, output_dir="raw_output", max_new_tokens=1024):

    os.makedirs(output_dir, exist_ok=True)

    result_df = df.copy()

    raw_output_files = []
    predictions = []
    runtimes = []
    models = []

    for idx, image in enumerate(result_df["image"]):

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        start_time = time.time()

        try:
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )

            generated = outputs[:, inputs["input_ids"].shape[1]:]

            prediction = processor.batch_decode(
                generated,
                skip_special_tokens=True
            )[0]

        except Exception as e:
            prediction = f"ERROR: {e}"

        runtime = time.time() - start_time

        filename = f"hunyuanocr_{idx:04d}.txt"

        with open(
            os.path.join(output_dir, filename),
            "w",
            encoding="utf-8"
        ) as f:
            f.write(prediction)

        raw_output_files.append(filename)
        predictions.append(prediction)
        runtimes.append(runtime)
        models.append("HunyuanOCR")

    result_df["raw_output"] = raw_output_files
    result_df["prediction"] = predictions
    result_df["runtime"] = runtimes
    result_df["model"] = models

    return result_df

In [ ]:
#MODEL : Noughat
"""transformers==4.38.2 \
huggingface_hub==0.20.3 \
timm==0.5.4 \
accelerate==0.27.2 \
datasets==2.18.0 \
sentencepiece \
nougat-ocr"""

In [ ]:
#Installation
!pip install \
transformers==4.38.2 \
huggingface_hub==0.20.3 \
timm==0.5.4 \
accelerate==0.27.2 \
datasets==2.18.0 \
sentencepiece \
nougat-ocr

In [ ]:
#Model Loading
import torch
from transformers import NougatProcessor, VisionEncoderDecoderModel
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = NougatProcessor.from_pretrained("facebook/nougat-base")

model = VisionEncoderDecoderModel.from_pretrained(
    "facebook/nougat-base"
).to(device)

model.eval()

In [ ]:
#Nougat Function
import os
import time
import pandas as pd
import torch
from tqdm.auto import tqdm

def run_nougat(
    df,
    model,
    processor,
    raw_output_dir="raw_output"
):

    os.makedirs(raw_output_dir, exist_ok=True)

    device = next(model.parameters()).device

    results = []

    model.eval()

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Running Nougat"):

        doc_id = row["doc_id"]
        image = row["image"]

        start = time.perf_counter()

        try:
            inputs = processor(
                image,
                return_tensors="pt"
            )

            pixel_values = inputs.pixel_values.to(device)

            with torch.no_grad():
                outputs = model.generate(
                    pixel_values,
                    max_new_tokens=1024,
                )

            prediction = processor.batch_decode(
                outputs,
                skip_special_tokens=True
            )[0]

        except Exception as e:
            prediction = f"ERROR: {e}"

        runtime = time.perf_counter() - start

        raw_filename = f"{doc_id}.txt"
        raw_path = os.path.join(raw_output_dir, raw_filename)

        with open(raw_path, "w", encoding="utf-8") as f:
            f.write(prediction)

        results.append({
            "doc_id": doc_id,
            "prediction": prediction,
            "runtime": runtime,
            "raw_output": raw_filename,
            "model": "Nougat"
        })

    return pd.DataFrame(results)

In [ ]:
#MODEL : Docling

In [ ]:
#Installation
pip install -U docling

In [ ]:
#Import
from docling.document_converter import DocumentConverter
converter = DocumentConverter()

In [ ]:
#Model loading
import os
import time
import tempfile
from pathlib import Path

from PIL import Image

from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

pipeline_options = PdfPipelineOptions()

# Disable expensive components
pipeline_options.do_table_structure = False
pipeline_options.do_formula_enrichment = False
pipeline_options.do_code_enrichment = False
pipeline_options.do_picture_classification = False
pipeline_options.do_picture_description = False

converter = DocumentConverter(
    format_options={
        InputFormat.IMAGE: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)


def run_docling(df):
    os.makedirs("raw_output", exist_ok=True)

    predictions = []
    runtimes = []
    raw_files = []

    # Create ONE temporary directory for the whole run
    with tempfile.TemporaryDirectory() as tmpdir:

        img_path = Path(tmpdir) / "temp_image.png"

        total = len(df)

        for i, row in df.iterrows():

            image = row["image"].copy()

            # Downsample large images
            image.thumbnail((1200, 1200), Image.Resampling.LANCZOS)

            image.save(img_path)

            start = time.perf_counter()

            result = converter.convert(img_path)

            markdown = result.document.export_to_markdown()

            runtime = time.perf_counter() - start

            raw_filename = f"docling_{i}.md"

            with open(
                os.path.join("raw_output", raw_filename),
                "w",
                encoding="utf-8",
            ) as f:
                f.write(markdown)

            predictions.append(markdown)
            runtimes.append(runtime)
            raw_files.append(raw_filename)

            print(
                f"[{i+1}/{total}] "
                f"{runtime:.2f}s"
            )

    out = df.copy()

    out["raw_output"] = raw_files
    out["prediction"] = predictions
    out["runtime"] = runtimes
    out["model"] = "Docling"

    return out

In [ ]:
#MODEL : Florence 2
"""!pip install -q \
transformers==4.49.0 \
tokenizers==0.21.0 \
accelerate \
sentencepiece"""

In [ ]:
#Installations
!pip install -q \
transformers==4.49.0 \
tokenizers==0.21.0 \
accelerate \
sentencepiece

In [ ]:
#Importing
import os
import time
import gc
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
#Model loading
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-base",
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-base",
    trust_remote_code=True,
    torch_dtype=torch_dtype
).to(device)

model.eval()

In [ ]:
#Florence 2 Function
def run_florence2_base(
    df,
    output_folder="florence2_raw_outputs",
    prompt="<OCR>",
    max_new_tokens=1024,
    num_beams=3,
):

    os.makedirs(output_folder, exist_ok=True)

    results = df.copy()

    results["raw_output"] = None
    results["prediction"] = None
    results["runtime"] = None
    results["model"] = "Florence-2-base"

    for idx, row in results.iterrows():

        image = row["image"]
        doc_id = row["doc_id"]

        start = time.perf_counter()

        try:

            inputs = processor(
                text=prompt,
                images=image,
                return_tensors="pt"
            )

            inputs = {
                k: v.to(device, dtype=torch_dtype if v.is_floating_point() else v.dtype)
                for k, v in inputs.items()
            }

            with torch.inference_mode():

                generated_ids = model.generate(
                    input_ids=inputs["input_ids"],
                    pixel_values=inputs["pixel_values"],
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    num_beams=num_beams,
                )

            raw_output = processor.batch_decode(
                generated_ids,
                skip_special_tokens=False
            )[0]

            parsed = processor.post_process_generation(
                raw_output,
                task=prompt,
                image_size=(image.width, image.height)
            )

            prediction = parsed.get(prompt, "")

        except Exception as e:

            raw_output = str(e)
            prediction = ""

        runtime = time.perf_counter() - start

        # Save raw output
        with open(
            os.path.join(output_folder, f"{doc_id}.txt"),
            "w",
            encoding="utf-8"
        ) as f:
            f.write(str(raw_output))

        # Update dataframe
        results.at[idx, "raw_output"] = raw_output
        results.at[idx, "prediction"] = prediction
        results.at[idx, "runtime"] = runtime

        # Free memory
        del inputs
        if "generated_ids" in locals():
            del generated_ids

        torch.cuda.empty_cache()
        gc.collect()

    return results

In [ ]:
#MODEL : SuryaOCR
"""
surya-ocr==0.13.0
transformers==4.38.2
Hugging Face Hub==0.36.2
Accelerate==1.14.0
Pillow==11.3.0"""

In [ ]:
#Installation
!pip install "surya-ocr==0.13.0"
!pip install "transformers==4.38.2"

In [ ]:
#Importing
import huggingface_hub
import accelerate
import PIL
import os
import gc
import time

import torch
from PIL import Image

from surya.detection import DetectionPredictor
from surya.recognition import RecognitionPredictor

In [ ]:
#Model loading
detector = DetectionPredictor()
recognizer = RecognitionPredictor()

In [ ]:
#Surya Function
def run_surya(
    df,
    output_folder="Surya_Raw_Outputs",
    languages=[["en"]]
):

    os.makedirs(output_folder, exist_ok=True)

    results = df.copy()

    results["raw_output"] = None
    results["prediction"] = None
    results["runtime"] = None
    results["model"] = "Surya OCR"

    for idx, row in results.iterrows():

        image = row["image"]
        doc_id = row["doc_id"]

        print(f"[{idx+1}/{len(results)}] {doc_id}")

        start = time.perf_counter()

        try:

            with torch.inference_mode():

                prediction = recognizer(
                    [image],
                    langs=languages,
                    det_predictor=detector
                )[0]

            raw_output = prediction

            text = "\n".join(
                line.text
                for line in prediction.text_lines
            )

        except Exception as e:

            raw_output = str(e)
            text = ""

        runtime = time.perf_counter() - start

        # Save raw output
        with open(
            os.path.join(output_folder, f"{doc_id}.txt"),
            "w",
            encoding="utf-8"
        ) as f:
            f.write(str(raw_output))

        # Update dataframe
        results.at[idx, "raw_output"] = raw_output
        results.at[idx, "prediction"] = text
        results.at[idx, "runtime"] = runtime
        results.at[idx, "model"] = "Surya OCR"

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

In [ ]:
#MODEL : DocTR
"""PyTorch           : 2.11.0+cu128
Hugging Face Hub  : 0.36.2
Accelerate        : 1.14.0
Pillow            : 10.4.0
"""

In [ ]:
#Installations
!pip install -q python-doctr[torch]

In [ ]:
#Importing
import os
import gc
import time
import torch
import numpy as np
from doctr.io import DocumentFile
from doctr.models import ocr_predictor

In [ ]:
#Model Loading
doctr_model = ocr_predictor(
    pretrained=True,
    detect_language=False,
    assume_straight_pages=True
)

In [ ]:
#DocTR Function
def run_doctr(
    df,
    output_folder="DocTR_Raw_Outputs"
):

    os.makedirs(output_folder, exist_ok=True)

    results = df.copy()

    results["raw_output"] = None
    results["prediction"] = None
    results["runtime"] = None
    results["model"] = "docTR"

    for idx, row in results.iterrows():

        image = np.array(row["image"])
        doc_id = row["doc_id"]

        print(f"[{idx+1}/{len(results)}] {doc_id}")

        start = time.perf_counter()

        try:

            with torch.inference_mode():

                prediction = doctr_model([image])

            raw_filename = f"{doc_id}.txt"

            # Plain OCR text
            text = prediction.render()

            # Save full raw object
            with open(
                os.path.join(output_folder, raw_filename),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(str(prediction.export()))

        except Exception as e:

            raw_filename = None
            text = ""
            print(f"Error on {doc_id}: {e}")

        runtime = time.perf_counter() - start

        results.at[idx, "raw_output"] = raw_filename
        results.at[idx, "prediction"] = text
        results.at[idx, "runtime"] = runtime
        results.at[idx, "model"] = "docTR"

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

In [ ]:
#MODEL : PaddleOCR-VL
"""paddlex[ocr]==3.7.0
paddlepaddle-gpu==3.3.0
transformer: 5.13.1
Pillow: 11.3.0
accelerate: 1.14.0"""

In [ ]:
#Installations
!pip install -U paddleocr
!pip install "paddlex[ocr]==3.7.0"
!python -m pip install paddlepaddle-gpu==3.3.0 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

In [ ]:
#Importing
import paddleocr
from paddleocr import PaddleOCR
from paddleocr import PaddleOCRVL
import os
import time
import numpy as np

In [ ]:
#Model loading
pipeline = PaddleOCRVL(
    device="gpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False
)

In [ ]:
#PaddleOCR-VL Function
def run_paddleocr_vl(df, raw_output_dir):

    os.makedirs(raw_output_dir, exist_ok=True)

    predictions = []
    runtimes = []
    raw_files = []

    for _, row in df.iterrows():

        image = np.array(row["image"].convert("RGB"))

        start = time.perf_counter()
        result = pipeline.predict(image)

        if isinstance(result, list):
            page = result[0]
        else:
            page = next(result)

        runtime = time.perf_counter() - start

        prediction = "\n".join(
            block.content
            for block in page["parsing_res_list"]
            if block.content and block.content.strip()
        )

        raw_filename = f"{row['doc_id']}.txt"

        with open(
            os.path.join(raw_output_dir, raw_filename),
            "w",
            encoding="utf-8",
        ) as f:

            for block in page["parsing_res_list"]:

                f.write("#################\n")
                f.write(f"label:\t{block.label}\n")
                f.write(f"bbox:\t{block.bbox}\n")
                f.write(f"content:\t{block.content}\n")
                f.write("#################\n\n")

        predictions.append(prediction)
        runtimes.append(runtime)
        raw_files.append(raw_filename)

        print(f"{row['doc_id']} completed in {runtime:.2f}s")

    out_df = df.copy()

    out_df["raw_output"] = raw_files
    out_df["prediction"] = predictions
    out_df["runtime"] = runtimes
    out_df["model"] = "PaddleOCR-VL-0.9B"

    return out_df

In [ ]:
#DeepSeekOCR (Taken from the official notebook)

In [ ]:
#Installing and Importing
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

import os
import time
import tempfile
import io
import contextlib
import gc
import torch

In [ ]:
#Model Loading
from huggingface_hub import snapshot_download

snapshot_download(
    "unsloth/DeepSeek-OCR",
    local_dir="deepseek_ocr",
)

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit", # Qwen 3 vision support
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
]

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit=False,                  # <-- Recommended
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=False,        # <-- Turn off compilation
    use_gradient_checkpointing=False,   # <-- Not needed for inference
)

In [ ]:
#DeepSeek Function
def run_deepseek_ocr(
    df,
    raw_output_dir,
    output_dir="deepseek_outputs",
):

    os.makedirs(raw_output_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    predictions = []
    runtimes = []
    raw_files = []

    total = len(df)

    for idx, row in enumerate(df.itertuples(index=False), start=1):

        raw_filename = f"{row.doc_id}.txt"
        raw_path = os.path.join(raw_output_dir, raw_filename)

        image_path = None

        try:
            # Save temporary image
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
                row.image.save(tmp.name)
                image_path = tmp.name

            buffer = io.StringIO()

            start = time.perf_counter()

            # Capture ONLY stdout (same as original)
            with contextlib.redirect_stdout(buffer):

                model.infer(
                    tokenizer,
                    prompt="<image>\nFree OCR.",
                    image_file=image_path,
                    output_path=output_dir,
                    base_size=1024,
                    image_size=640,
                    crop_mode=False,
                    save_results=False,
                    test_compress=False,
                )

            runtime = time.perf_counter() - start

            prediction = buffer.getvalue().strip()

            # Remove console noise
            prediction_lines = []

            for line in prediction.splitlines():

                line = line.strip()

                if (
                    not line
                    or line.startswith("=")
                    or line.startswith("BASE:")
                    or line.startswith("PATCHES:")
                    or line.startswith("image:")
                    or line.startswith("other:")
                    or "save results" in line.lower()
                ):
                    continue

                prediction_lines.append(line)

            prediction = "\n".join(prediction_lines)

            with open(raw_path, "w", encoding="utf-8") as f:
                f.write(prediction)

            predictions.append(prediction)
            runtimes.append(runtime)
            raw_files.append(raw_filename)

            print(f"[{idx}/{total}] Done {row.doc_id} ({runtime:.2f}s)")

        except Exception as e:

            print(f"[{idx}/{total}] FAILED {row.doc_id}: {e}")

            predictions.append("")
            runtimes.append(None)
            raw_files.append(raw_filename)

        finally:

            if image_path is not None and os.path.exists(image_path):
                os.remove(image_path)

            gc.collect()
            torch.cuda.empty_cache()

    out_df = df.copy()

    out_df["raw_output"] = raw_files
    out_df["prediction"] = predictions
    out_df["runtime"] = runtimes
    out_df["model"] = "DeepSeek-OCR-3B"

    return out_df

In [ ]:
#MODEL : GOTOCR2.0
"""
Transformers == 4.45.2
verovio-6.2.1
"""

In [ ]:
#Installation
!pip install transformers==4.45.2
!pip install -U accelerate sentencepiece

In [ ]:
#Importing
import os
import gc
import time
import tempfile
import torch
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from PIL import Image
import time

In [ ]:
#Model Loading

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    "stepfun-ai/GOT-OCR2_0",
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    "stepfun-ai/GOT-OCR2_0",
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

In [ ]:
#GotOCR Function
def run_gotocr(
    df,
    raw_output_dir="raw_outputs",
):

    os.makedirs(raw_output_dir, exist_ok=True)

    predictions = []
    runtimes = []
    raw_outputs = []

    total = len(df)

    for idx, row in enumerate(df.itertuples(index=False), start=1):

        raw_filename = f"{row.doc_id}.txt"
        raw_path = os.path.join(raw_output_dir, raw_filename)

        # Resume support
        if os.path.exists(raw_path):

            with open(raw_path, "r", encoding="utf-8") as f:
                prediction = f.read().strip()

            predictions.append(prediction)
            runtimes.append(None)
            raw_outputs.append(raw_filename)

            print(f"[{idx}/{total}] Skipping {row.doc_id}")

            continue

        image_path = None

        try:

            with tempfile.NamedTemporaryFile(
                suffix=".png",
                delete=False
            ) as tmp:

                row.image.save(tmp.name)
                image_path = tmp.name

            start = time.perf_counter()

            prediction = model.chat(
                tokenizer,
                image_path,
                ocr_type="ocr"
            )

            runtime = time.perf_counter() - start

            prediction = prediction.strip()

            with open(raw_path, "w", encoding="utf-8") as f:
                f.write(prediction)

            predictions.append(prediction)
            runtimes.append(runtime)
            raw_outputs.append(raw_filename)

            print(
                f"[{idx}/{total}] Done {row.doc_id} "
                f"({runtime:.2f}s)"
            )

        except Exception as e:

            print(f"[{idx}/{total}] FAILED {row.doc_id}: {e}")

            predictions.append("")
            runtimes.append(None)
            raw_outputs.append(raw_filename)

        finally:

            if image_path is not None and os.path.exists(image_path):
                os.remove(image_path)

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    results = df.copy()

    results["raw_output"] = raw_outputs
    results["prediction"] = predictions
    results["runtime"] = runtimes
    results["model"] = "GOT-OCR2"

    return results

In [ ]:
#MODEL : PPOCR-V6
"""
paddleocr                             3.0.3
paddlepaddle                          3.0.0
paddlex                               3.0.3"""
!pip install paddlex==3.0.3

In [ ]:
#Importing
import os
import time
import numpy as np
import pandas as pd
import PIL
from paddleocr import PaddleOCR

In [ ]:
#Model Loading
ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    lang="en"
)

In [ ]:
#PPOCR Function
def run_ppocrv6(df, raw_output_dir):

    os.makedirs(raw_output_dir, exist_ok=True)

    predictions = []
    runtimes = []
    raw_files = []

    for _, row in df.iterrows():

        pil_img = row["image"]
        img = np.array(pil_img)

        start = time.perf_counter()

        try:
            result = ocr.predict(img)

            runtime = time.perf_counter() - start

            raw_filename = f"{row['doc_id']}.txt"
            raw_path = os.path.join(raw_output_dir, raw_filename)

            with open(raw_path, "w", encoding="utf-8") as f:
                f.write(str(result))

            prediction = ""

            try:
                texts = []

                if (
                    isinstance(result, list)
                    and len(result) > 0
                    and "rec_texts" in result[0]
                ):
                    texts = result[0]["rec_texts"]

                prediction = "\n".join(texts)

            except Exception:
                prediction = ""

        except Exception as e:

            runtime = time.perf_counter() - start

            raw_filename = f"{row['doc_id']}.txt"
            raw_path = os.path.join(raw_output_dir, raw_filename)

            with open(raw_path, "w", encoding="utf-8") as f:
                f.write(str(e))

            prediction = ""

        predictions.append(prediction)
        runtimes.append(runtime)
        raw_files.append(raw_filename)

    out_df = df.copy()

    out_df["raw_output"] = raw_files
    out_df["prediction"] = predictions
    out_df["runtime"] = runtimes
    out_df["model"] = "PP-OCRv6"

    return out_df

In [ ]:
#MODEL : Qwen2_5_VL
"""
av-18.0.0
pillow-12.3.0
qwen-vl-utils-0.0.14
transformers-5.14.1"""

In [ ]:
#Installation
!pip install -U transformers accelerate qwen-vl-utils pillow

In [ ]:
#Importing
import os
import time
import torch
import pandas as pd
from PIL import Image

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
)

In [ ]:
#Model Loading
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

In [ ]:
def build_messages(image):

    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": "You are an OCR system.Read the document and output ONLY the recognized text.Do not summarize.Do not explain.Do not describe the layout.Do not add Markdown.Preserve line breaks whenever possible..",
                },
            ],
        }
    ]

In [ ]:
#Qwen 2.5 - VL Function
def run_qwen25vl(df, raw_output_dir):

    os.makedirs(raw_output_dir, exist_ok=True)

    predictions = []
    runtimes = []
    raw_files = []

    for _, row in df.iterrows():

        image = row["image"].convert("RGB")

        messages = build_messages(image)

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = processor(
            text=[text],
            images=[image],
            return_tensors="pt",
        )

        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        start = time.perf_counter()

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=False,
        )

        runtime = time.perf_counter() - start

        prediction = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )[0]

        raw_filename = f"{row['doc_id']}.txt"

        with open(
            os.path.join(raw_output_dir, raw_filename),
            "w",
            encoding="utf-8",
        ) as f:
            f.write(prediction)

        predictions.append(prediction)
        runtimes.append(runtime)
        raw_files.append(raw_filename)

    out = df.copy()

    out["raw_output"] = raw_files
    out["prediction"] = predictions
    out["runtime"] = runtimes
    out["model"] = "Qwen2.5-VL-3B-Instruct"

    return out

In [ ]:
#MODEL : MinerU2.5
"""httpx-retries-0.6.0
 huggingface_hub-0.36.2
 loguru-0.7.3
  mineru-vl-utils-1.0.5
  transformers-4.57.6"""

In [ ]:
#Installation
pip install "mineru-vl-utils[transformers]"

In [ ]:
#Importing
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from mineru_vl_utils import MinerUClient
from loguru import logger
logger.remove()
import os
import time
import gc
import torch
import pandas as pd

In [ ]:
#Model Loading
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "opendatalab/MinerU2.5-2509-1.2B",
    torch_dtype="auto",
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(
    "opendatalab/MinerU2.5-2509-1.2B",
    use_fast=True,
)

client = MinerUClient(
    backend="transformers",
    model=model,
    processor=processor,
)

In [ ]:
#MinerU2.5 Function
def run_mineru(
    df,
    client,
    raw_output_dir,
):

    os.makedirs(raw_output_dir, exist_ok=True)

    predictions = []
    runtimes = []
    raw_files = []

    total = len(df)

    for idx, row in enumerate(df.itertuples(index=False), start=1):

        raw_filename = f"{row.doc_id}.txt"
        raw_path = os.path.join(raw_output_dir, raw_filename)

        try:

            start = time.perf_counter()

            result = client.two_step_extract(row.image)

            runtime = time.perf_counter() - start

            # Concatenate all textual blocks
            prediction = "\n".join(
                block["content"]
                for block in result
                if block.get("content")
            ).strip()

            with open(raw_path, "w", encoding="utf-8") as f:
                f.write(prediction)

            predictions.append(prediction)
            runtimes.append(runtime)
            raw_files.append(raw_filename)

            print(
                f"[{idx}/{total}] Done {row.doc_id} "
                f"({runtime:.2f}s)"
            )

        except Exception as e:

            print(f"[{idx}/{total}] FAILED {row.doc_id}: {e}")

            predictions.append("")
            runtimes.append(None)
            raw_files.append(raw_filename)

        finally:

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    out_df = df.copy()

    out_df["raw_output"] = raw_files
    out_df["prediction"] = predictions
    out_df["runtime"] = runtimes
    out_df["model"] = "MinerU2.5-1.2B"

    return out_df

In [ ]:
#MODEL: InternVL2-1B
"""accelerate-1.9.0
decord-0.6.0
huggingface_hub-0.36.2
tokenizers-0.21.4
transformers-4.55.4"""

In [ ]:
#Installations
!pip install \
transformers==4.55.4 \
accelerate==1.9.0 \
timm \
huggingface_hub \
sentencepiece \
einops \
pillow \
numpy \
torchvision \
decord

In [ ]:
#Importing
import math
import numpy as np
import torch
import torchvision.transforms as T

from PIL import Image
from torchvision.transforms.functional import InterpolationMode

from transformers import AutoTokenizer, AutoModel

In [ ]:
#Model loading
path = "OpenGVLab/InternVL2-1B"
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_flash_attn=False,
    trust_remote_code=True).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(
    path,
    trust_remote_code=True,
    use_fast=False,
)

print(type(tokenizer))

In [ ]:
#InternVL2 Function
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

In [ ]:
def load_image(image, input_size=448, max_num=12):

    image = image.convert("RGB")

    transform = build_transform(input_size=input_size)

    images = dynamic_preprocess(
        image,
        image_size=input_size,
        use_thumbnail=True,
        max_num=max_num,
    )

    pixel_values = [transform(img) for img in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

In [ ]:
def benchmark_internvl(
    df,
    model,
    tokenizer,
    output_csv="internvl_results.csv",
    raw_output_dir="internvl_raw",
    save_every=25,
):
    os.makedirs(raw_output_dir, exist_ok=True)

    # Resume if previous results exist
    if os.path.exists(output_csv):
        results_df = pd.read_csv(output_csv)
        completed = set(results_df["doc_id"].astype(str))
        results = results_df.to_dict("records")
        print(f"Resuming from {len(completed)} completed documents.")
    else:
        completed = set()
        results = []

    total = len(df)

    for idx, row in df.iterrows():

        doc_id = str(row["doc_id"])

        if doc_id in completed:
            continue

        image = row["image"]

        try:
            start = time.time()

            pixel_values = load_image(image)
            pixel_values = pixel_values.to(next(model.parameters()).dtype).cuda()

            generation_config = {
                "max_new_tokens": 1024,
                "do_sample": False,
            }

            prediction = model.chat(
                tokenizer,
                pixel_values,
                "<image>\nRead all text in this document exactly as written.",
                generation_config,
            )

            runtime = time.time() - start

            raw_filename = f"{doc_id}.txt"

            with open(
                os.path.join(raw_output_dir, raw_filename),
                "w",
                encoding="utf-8",
            ) as f:
                f.write(prediction)

            results.append(
                {
                    "doc_id": doc_id,
                    "prediction": prediction,
                    "runtime": runtime,
                    "raw_output": raw_filename,
                    "model": "InternVL2-1B",
                }
            )

            completed.add(doc_id)

            print(f"[{len(completed)}/{total}] {doc_id}")

            # Save every N documents
            if len(completed) % save_every == 0:
                pd.DataFrame(results).to_csv(output_csv, index=False)

            torch.cuda.empty_cache()

        except Exception as e:
            print(f"[ERROR] {doc_id}: {e}")

    # Final save
    pd.DataFrame(results).to_csv(output_csv, index=False)

    print("\nBenchmark complete!")

    return pd.DataFrame(results)

In [ ]:
#MODEL : Tesseract

In [ ]:
#Installations
!sudo apt-get update
!sudo apt-get install -y tesseract-ocr
!pip install -q pytesseract

In [ ]:
#Importing
import time
import pandas as pd
import pytesseract

In [ ]:
#Tesseract Function
def run_tesseract(dataset_df, lang = "eng", config=""):

    # Make a copy so the original dataframe is untouched
    results_df = dataset_df.copy()

    raw_outputs = []
    predictions = []
    runtimes = []

    for _, row in results_df.iterrows():

        image = row["image"]

        start = time.perf_counter()

        raw_output = pytesseract.image_to_string(
            image,
            lang=lang,
            config=config
        )

        runtime = time.perf_counter() - start

        # For Tesseract the raw output is already plain text
        prediction = raw_output

        raw_outputs.append(raw_output)
        predictions.append(prediction)
        runtimes.append(runtime)

    results_df["raw_output"] = raw_outputs
    results_df["prediction"] = predictions
    results_df["runtime"] = runtimes
    results_df["model"] = "Tesseract"

    return results_df

In [ ]:
#MODEL : EasyOCR

In [ ]:
#Installation, importing and loading model
!pip install -q easyocr
import easyocr
import numpy as np
easyocr_reader = easyocr.Reader(["en"], gpu=True)

In [ ]:
#EasyOCR function
def run_easyocr(dataset_df):
    results_df = dataset_df.copy()
    raw_outputs = []
    predictions = []
    runtimes = []

    for _, row in results_df.iterrows():

        image = np.array(row["image"])
        start = time.perf_counter()
        raw_output = easyocr_reader.readtext(image)
        runtime = time.perf_counter() - start

        prediction = "\n".join([item[1] for item in raw_output])

        raw_outputs.append(raw_output)
        predictions.append(prediction)
        runtimes.append(runtime)

    results_df["raw_output"] = raw_outputs
    results_df["prediction"] = predictions
    results_df["runtime"] = runtimes
    results_df["model"] = "EasyOCR"

    return results_df

In [ ]:
#At the end of each model, it was run on the df and the dataframe was saved as a csv while the raw outputs were downloaded as zip files.